# 4.5 The Full Hodge-Laplacian

Reproduces Table 4.8 (full-Hodge eigenvalues at $\beta = 0$ against the union of the Dirichlet and Neumann spectra of $-\Delta$ on $(0,\pi)^2$) and Table 4.9 (half- versus full-Hodge spectra under $\beta_2$ and $\beta_3$). Imports the shared module `spectral_common.py` from this folder. Run top to bottom in a kernel with Firedrake + SLEPc + MUMPS; this section writes no figures (the imports cell still creates `./figures`, which the other notebooks use). The code writes the polynomial degree as `r`; the thesis calls it $p$.


## Setup


Imports, `FIGDIR`, the `show` table helper.


In [1]:
from __future__ import annotations

import gc
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Markdown

from firedrake import *
from firedrake.pyplot import tripcolor, quiver

from petsc4py import PETSc

import spectral_common as sc
from spectral_common import (crisscross_mesh, solve_pencil, rates, assign,
                             agreeing_digits, spread, L_DOMAIN, KAPPA_LOG)

# `from firedrake import *` exports a `logging` of its own, so the standard
# library one has to be re-imported under another name afterwards.
import logging as stdlogging
stdlogging.getLogger("firedrake").setLevel(stdlogging.ERROR)
stdlogging.getLogger("tsfc").setLevel(stdlogging.ERROR)

FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True)

# Everything the LaTeX block above does not set: fonts and sizes come from there.
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})

# The conditioning tally at the end runs to a few dozen rows; pandas would elide
# the middle of it, which is exactly the part a tally exists to show.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


def show(df, caption=None, **kw):
    """`sc.fmt_table` wired to this notebook's display function."""
    return sc.fmt_table(df, caption=caption,
                        display_fn=lambda c: display(Markdown(f"**{c}**")), **kw)


print(f"PETSc scalars: {PETSc.ScalarType.__name__}")

PETSc scalars: float64


Mixed formulations $B(\mathcal{N}^{\mathrm{I}}_r)$, $B(\mathcal{N}^{\mathrm{II}}_r)$ and `build_pencil`; `full=False` drops the $(p,q)$ block and gives the half-Hodge problem.


In [2]:
def cross2d(a, b):
    r"""Scalar 2D cross product $(a\times b)_z = a_1b_2 - a_2b_1$.

    Antisymmetric, so the ordering is load-bearing: the induction term needs
    $u\times\beta$, the proxy of the contraction $\iota^1_\beta u$.
    """
    return a[0] * b[1] - a[1] * b[0]


FORMS = ["B(N1)", "B(N2)"]
TEX = {"B(N1)": r"$B(\mathcal{N}^{\mathrm{I}}_r)$",
       "B(N2)": r"$B(\mathcal{N}^{\mathrm{II}}_r)$"}
TAG = {"B(N1)": "BN1", "B(N2)": "BN2"}
MARK = {"B(N1)": "o", "B(N2)": "s"}
LS = {"B(N1)": "-", "B(N2)": "--"}


def space(form, mesh, r=1):
    r"""The mixed space $\Lambda^1_h\times\Lambda^0_h$ of one pair."""
    if form == "B(N1)":
        return FunctionSpace(mesh, "N1curl", r) * FunctionSpace(mesh, "CG", r)
    return FunctionSpace(mesh, "N2curl", r) * FunctionSpace(mesh, "CG", r + 1)


def build_pencil(form, mesh, r=1, beta=None, eps=Constant(1.0), full=True):
    r"""Assemble $(A, M)$.  ``full=False`` deletes the $(p,q)$ block: half-Hodge.

    Three edits separate the two problems and they are made together, because
    separately none of them is meaningful: the mass block $(p,q)$ turns the
    multiplier into $p = \delta u$; the $\varepsilon$ on $(\nabla p, v)$ puts
    that term inside the Laplacian instead of beside it; and the contraction
    $(p\beta, v)$ is the second half of Cartan's formula, invisible whenever
    $\mathrm{d}u = 0$.

    $M$ is singular in both problems -- the $p$ block carries no mass -- which is
    what the shift-and-invert transformation is for.
    """
    W = space(form, mesh, r)
    (u, p) = TrialFunctions(W)
    (v, q) = TestFunctions(W)
    e = eps

    a = e * inner(curl(u), curl(v)) * dx                # eps (d u, d v)
    # (d^0 p, v): carries eps in the full problem, where it is part of the
    # Laplacian, and none in the half problem, where p is a Lagrange multiplier.
    a += -(e if full else Constant(1.0)) * inner(grad(p), v) * dx
    a += inner(u, grad(q)) * dx                         # -(u, d^0 q)
    if full:
        a += inner(p, q) * dx                           # (p, q): the whole difference
    if beta is not None:
        a += inner(cross2d(u, beta), curl(v)) * dx      # (iota^1_beta u, d v)
        if full:
            a += p * inner(beta, v) * dx                # (iota^2_beta p, v)

    # On an H(curl) space a DirichletBC constrains the TANGENTIAL trace, which is
    # the perfectly-conducting condition; p = 0 puts the second component in H^1_0.
    bcs = [DirichletBC(W.sub(0), Constant((0.0, 0.0)), "on_boundary"),
           DirichletBC(W.sub(1), Constant(0.0), "on_boundary")]
    # weight=0.0 gives the constrained rows a zero mass diagonal, so the boundary
    # modes go to infinity rather than to a spurious lambda = 1.
    return (assemble(a, bcs=bcs).petscmat,
            assemble(inner(u, v) * dx, bcs=bcs, weight=0.0).petscmat, W)

Criss-cross mesh of $(0,\pi)^2$ (`mesh_for`), the shift-and-invert driver `solve_form` ($\tau = 0.9/R_m$ by default) and the conditioning columns `kappa_cols`.


In [3]:
def h_of(N):
    return L_DOMAIN / N


#: Meshes are reused across studies, so every mesh in the notebook comes from here.
_MESH_CACHE = {}


def mesh_for(N):
    if N not in _MESH_CACHE:
        _MESH_CACHE[N] = crisscross_mesh(N)
    return _MESH_CACHE[N]


def solve_form(form, N=None, r=1, field=None, Rm=1.0, full=True, mesh=None,
               tau=None, nev=40, n_eigs=None, cond=True, **meta):
    r"""Assemble and solve one configuration.  ``tau`` defaults to $0.9\varepsilon$."""
    mesh = mesh_for(N) if mesh is None else mesh
    beta = None if field is None else FIELDS[field](mesh)
    A, M, W = build_pencil(form, mesh, r, beta=beta, eps=Constant(1.0 / Rm),
                           full=full)
    return solve_pencil(A, M, (0.9 / Rm) if tau is None else tau, nev=nev,
                        n_eigs=n_eigs, space=W, cond=cond, form=form, N=N, r=r,
                        Rm=Rm, field=field, full=full, **meta)


#: Column names for the conditioning, so that every table in the notebook spells
#: them the same way and a format rule written once matches everywhere.
KAP_HAGER, KAP_MUMPS, KAP_INFOG = r"Hager $\kappa_1$", "MUMPS COND1", "INFOG(1)"


def kappa_cols(res):
    r"""Both estimates of $\kappa_1(A - \tau M)$, as table columns.

    Every solve in this notebook is run with ``cond=True`` and every table that
    reports a result reports these alongside it.  `MUMPS COND1` is the
    componentwise Arioli--Demmel--Duff estimate produced by the factorisation
    that the shift-and-invert transformation had to perform anyway, so it costs
    nothing beyond an error analysis and a dummy solve; `Hager` is the normwise
    $\|S\|_1\|S^{-1}\|_1$ from an independent sparse LU, carried as a check on
    it.  `INFOG(1) = 0` is MUMPS reporting that the factorisation succeeded.

    The normwise estimate reads `--` on the two $r = 3$ baseline pencils and only
    there: its second sparse LU roughly doubles the peak memory of a solve, and
    at $N = 64$, $r = 3$ that is the difference between the notebook running and
    the kernel being killed.  MUMPS' `COND1` costs nothing extra and is reported
    for every solve without exception.
    """
    return {KAP_HAGER: res.kappa[r"Hager $\kappa_1$"],
            KAP_MUMPS: res.kappa["MUMPS COND1"],
            KAP_INFOG: res.kappa["INFOG(1)"]}


#: The format rules the conditioning columns need, in every table that has them.
KAP_RULES = [(KAP_HAGER, "{:.3e}"), (KAP_MUMPS, "{:.3e}"), (KAP_INFOG, "{:.0f}")]

Winds $\beta_2 = (x,-y)$, $\beta_3 = (\partial_y\psi, \partial_x\psi)$ (and $\beta_{11}$, unused here), each rescaled to unit r.m.s.; the stored table measures their curl and divergence norms on the $N = 32$ mesh.


In [4]:
def unit_wind(mesh, b):
    r"""$\beta$ rescaled to unit r.m.s. magnitude over the domain."""
    vol = assemble(Constant(1.0) * dx(domain=mesh))
    rms = float(np.sqrt(assemble(inner(b, b) * dx(domain=mesh)) / vol))
    return b / Constant(rms)


@dataclass(frozen=True)
class Field:
    """A named velocity field together with the properties that matter here."""
    key: str
    tex: str
    regularity: str
    solenoidal: bool
    _build: Callable

    def __call__(self, mesh, raw=False):
        b = self._build(mesh)
        return b if raw else unit_wind(mesh, b)


def _b2(mesh):
    r"""$\beta_2 = (x, -y)$: linear strain.  Curl-free and divergence-free."""
    x, y = SpatialCoordinate(mesh)
    return as_vector([x, -y])


def _b3(mesh):
    r"""$\beta_3 = (\psi_y, \psi_x)$ with $\psi = \sin 2x \sin 2y$.

    Curl-free -- $\psi_{xx} - \psi_{yy} = 0$ for this $\psi$ -- so King's theorem
    applies; but $\psi_{xy} \neq 0$, so unlike every other smooth field in the
    table it is *not* solenoidal, with $\nabla\!\cdot\beta = 8\cos 2x\cos 2y$.
    Written through UFL derivatives of $\psi$ rather than expanded by hand, so
    that the identity $\beta = (\psi_y, \psi_x)$ is visible in the code.
    """
    x, y = SpatialCoordinate(mesh)
    psi = sin(2 * x) * sin(2 * y)
    return as_vector([psi.dx(1), psi.dx(0)])


def _b11(mesh, delta=1e-12):
    r"""$\beta_{11} = (\ln r_0, \ln r_0)$: in every $L^p$, $p<\infty$, in no $L^\infty$."""
    x, y = SpatialCoordinate(mesh)
    xc = yc = Constant(L_DOMAIN / 2)
    r0 = sqrt((x - xc) ** 2 + (y - yc) ** 2 + Constant(delta))
    return as_vector([ln(r0), ln(r0)])


FIELDS = {
    "b2": Field("b2", r"$(x,\,-y)$", r"$C^\infty$", True, _b2),
    "b3": Field("b3", r"$(\partial_y\psi,\ \partial_x\psi)$", r"$C^\infty$",
                False, _b3),
    "b11": Field("b11", r"$(\ln r_0,\ \ln r_0)$",
                 r"$\bigcap_{p<\infty}L^p\setminus L^\infty$", False, _b11),
}

N_MEASURE = 32
m_meas = mesh_for(N_MEASURE)
rows = []
for key, f in FIELDS.items():
    b = f(m_meas)
    vol = assemble(Constant(1.0) * dx(domain=m_meas))
    rows.append({
        "field": key,
        r"$\beta$": f.tex,
        "regularity": f.regularity,
        r"$\|\operatorname{curl}\beta\|_{L^2}$":
            float(sqrt(assemble((b[1].dx(0) - b[0].dx(1)) ** 2 * dx))),
        r"$\|\nabla\!\cdot\beta\|_{L^2}$":
            float(sqrt(assemble((b[0].dx(0) + b[1].dx(1)) ** 2 * dx))),
        r"$\nabla\!\cdot\beta = 0$": "yes" if f.solenoidal else "no",
        "r.m.s.": float(np.sqrt(assemble(inner(b, b) * dx) / vol)),
    })
df_fields = pd.DataFrame(rows).set_index("field")
show(df_fields, default="{}",
     rules=[(r"$\|\operatorname{curl}\beta\|_{L^2}$", "{:.3e}"),
            (r"$\|\nabla\!\cdot\beta\|_{L^2}$", "{:.3e}"),
            ("r.m.s.", "{:.6f}")],
     caption=rf"The three winds after normalisation, measured on the $N = {N_MEASURE}$ "
             r"mesh. $\beta_2$ and $\beta_3$ are curl-free to round-off and differ "
             r"only in the divergence; $\beta_{11}$ is neither curl-free nor "
             r"solenoidal, and its two norms are resolution-dependent by "
             r"construction -- that is what leaving $W^{1,\infty}$ means -- so they "
             r"are quoted at this $N$ alone")

**The three winds after normalisation, measured on the $N = 32$ mesh. $\beta_2$ and $\beta_3$ are curl-free to round-off and differ only in the divergence; $\beta_{11}$ is neither curl-free nor solenoidal, and its two norms are resolution-dependent by construction -- that is what leaving $W^{1,\infty}$ means -- so they are quoted at this $N$ alone**

,$\beta$,regularity,$\|\operatorname{curl}\beta\|_{L^2}$,$\|\nabla\!\cdot\beta\|_{L^2}$,$\nabla\!\cdot\beta = 0$,r.m.s.
field,,,,,,
b2,"$(x,\,-y)$",$C^\infty$,0.000e+00,0.000e+00,yes,1.000000
b3,"$(\partial_y\psi,\ \partial_x\psi)$",$C^\infty$,0.000e+00,8.886e+00,no,1.000000
b11,"$(\ln r_0,\ \ln r_0)$",$\bigcap_{p<\infty}L^p\setminus L^\infty$,1.154e+01,1.154e+01,no,1.000000


## Table 4.8: Full-Hodge eigenvalues on (0,π)² at Rm = 1


$\beta = 0$, $R_m = 1$, criss-cross $N = 48$, $p = 2$ (`R_EXACT`), $\tau = 0.9$, `nev = 60`; the first 12 eigenvalues are tabulated against `exact_full` (Dirichlet $\cup$ Neumann spectra with multiplicity) and the thesis prints the first five. The $\kappa_1(A - \tau M)$ row of the thesis table is the `MUMPS COND1` column of the second table.


In [5]:
N_EXACT, R_EXACT, N_EX_SHOW = 48, 2, 12


def exact_full(n, kmax=12):
    r"""The first `n` values of $\sigma(\mathrm{Dirichlet}) \cup \sigma(\mathrm{Neumann})$."""
    dirichlet = [m * m + k * k for m in range(1, kmax) for k in range(1, kmax)]
    neumann = [m * m + k * k for m in range(0, kmax) for k in range(0, kmax)
               if (m, k) != (0, 0)]
    return np.array(sorted(dirichlet + neumann))[:n]


ex = exact_full(N_EX_SHOW)
rows = []
zero_runs = {f: solve_form(f, N_EXACT, R_EXACT, field=None, nev=60,
                           n_eigs=N_EX_SHOW, cond=True) for f in FORMS}
for j in range(N_EX_SHOW):
    row = {"j": j + 1, "exact": float(ex[j])}
    for f in FORMS:
        got = zero_runs[f].values[j]
        row[TEX[f]] = got
        row[f"err {TAG[f]}"] = abs(got - ex[j]) / ex[j]
    rows.append(row)
df_exact = pd.DataFrame(rows).set_index("j")
display(show(df_exact, default="{:.8f}",
             rules=[("exact", "{:.0f}"), ("err", "{:.2e}")],
             caption=rf"$\beta = 0$, $N = {N_EXACT}$, $r = {R_EXACT}$: the full Hodge "
                     r"spectrum against the union with multiplicity of the Dirichlet "
                     r"and Neumann spectra of $-\Delta$ on $(0,\pi)^2$"))

# The windless pencil's conditioning, on the same footing as every other solve in
# the notebook.  Nothing here is reported without it.
df_exact_kap = pd.DataFrame(
    [{"formulation": f, "dof": zero_runs[f].size, **kappa_cols(zero_runs[f])}
     for f in FORMS]).set_index("formulation")
show(df_exact_kap, default="{:.3e}", rules=[("dof", "{:.0f}")] + KAP_RULES,
     caption=rf"Conditioning of $A - \tau M$ for those same two solves, "
             rf"$N = {N_EXACT}$, $r = {R_EXACT}$, $\tau = 0.9$")

**$\beta = 0$, $N = 48$, $r = 2$: the full Hodge spectrum against the union with multiplicity of the Dirichlet and Neumann spectra of $-\Delta$ on $(0,\pi)^2$**

,exact,$B(\mathcal{N}^{\mathrm{I}}_r)$,err BN1,$B(\mathcal{N}^{\mathrm{II}}_r)$,err BN2
j,,,,,
1,1,1.00000000,2.94e-09,1.00000000,4.78e-09
2,1,1.00000000,2.94e-09,1.00000000,4.78e-09
3,2,2.00000000,1.56e-09,2.00000000,5.22e-13
4,2,2.00000004,2.17e-08,2.00000003,1.53e-08
5,4,4.00000019,4.70e-08,4.00000031,7.64e-08
6,4,4.00000019,4.70e-08,4.00000031,7.64e-08
7,5,5.00000016,3.26e-08,5.00000000,1.41e-11
8,5,5.00000016,3.26e-08,5.00000000,1.41e-11
9,5,5.00000089,1.78e-07,5.00000052,1.04e-07


**Conditioning of $A - \tau M$ for those same two solves, $N = 48$, $r = 2$, $\tau = 0.9$**

,dof,Hager $\kappa_1$,MUMPS COND1,INFOG(1)
formulation,,,,
B(N1),64897,1.703e+08,1.395e+05,0
B(N2),111169,2.920e+08,1.005e+05,0


## Table 4.9: Half- versus full-Hodge spectra for β2 and β3


Fields $\beta_2$, $\beta_3$; both formulations, half (`full=False`) and full (`full=True`); $R_m = 1$, $N = 32$, $p = 1$, $\tau = 0.9$, `nev = 120`, first 8 eigenvalues shown. The thesis prints the $B(\mathcal{N}^{\mathrm{I}}_1)$ rows only, with `--` in the half column where the full problem's value is marked *new in full*.


In [6]:
N_C, R_C, N_C_SHOW = 32, 1, 8
CFIELDS = ["b2", "b3"]

comp = {}
for key in CFIELDS:
    for form in FORMS:
        for full in (False, True):
            comp[(key, form, full)] = solve_form(
                form, N_C, R_C, field=key, full=full, nev=120, cond=True)

rows = []
for key in CFIELDS:
    for form in FORMS:
        half, fullr = comp[(key, form, False)], comp[(key, form, True)]
        hv, fv = half.values, fullr.values
        for j in range(N_C_SHOW):
            # Which of the full problem's values are new, i.e. not in the half
            # spectrum?  Those are the modes the second half of the Laplacian adds.
            d_new = min(abs(fv[j] - z) for z in hv)
            rows.append({"field": key, "formulation": form, "j": j + 1,
                         r"$\lambda_j$ half": hv[j],
                         r"$\lambda_j$ full": fv[j],
                         r"$\mathrm{dist}(\lambda_j^{\rm half},\sigma^{\rm full}_h)$":
                             min(abs(hv[j] - w) for w in fv),
                         "new in full": "yes" if d_new > 1e-8 * abs(fv[j]) else "--"})
df_comp = pd.DataFrame(rows).set_index(["field", "formulation", "j"])
show(df_comp, default="{:.7f}",
     rules=[(r"$\mathrm{dist}(\lambda_j^{\rm half},\sigma^{\rm full}_h)$", "{:.2e}"),
            ("new in full", "{}")],
     caption=rf"$N = {N_C}$, $r = {R_C}$, $R_m = 1$. Every half eigenvalue is a full "
             r"eigenvalue to round-off, so $\sigma(\mathrm{half}_h)\subseteq"
             r"\sigma(\mathrm{full}_h)$ holds under the compressible wind exactly as "
             r"under the solenoidal one; the values marked *new in full* are the modes "
             r"the $\mathrm{d}\delta$ half of the Laplacian contributes")

**$N = 32$, $r = 1$, $R_m = 1$. Every half eigenvalue is a full eigenvalue to round-off, so $\sigma(\mathrm{half}_h)\subseteq\sigma(\mathrm{full}_h)$ holds under the compressible wind exactly as under the solenoidal one; the values marked *new in full* are the modes the $\mathrm{d}\delta$ half of the Laplacian contributes**

$\lambda_j$ half $\lambda_j$ full  \
field formulation j                                     
b2    B(N1)       1        0.9095426        0.9095426   
                  2        1.2992149        1.2992149   
                  3        2.2081459        2.2081459   
                  4        3.9265215        2.2094933   
                  5        4.3156083        3.9265215   
                  6        5.2227837        4.3156083   
                  7        5.2230502        5.2227837   
                  8        8.2317069        5.2230502   
      B(N2)       1        0.9096288        0.9096288   
                  2        1.2994772        1.2994772   
                  3        2.2095384        2.2089351   
                  4        3.9292441        2.2095384   
                  5        4.3190923        3.9292441   
                  6        5.2304552        4.3190923   
                  7        5.2307224        5.2251371   
                  8        8.2563989        5.2251374   
b3    B(N1)       1        0.9218051        0.9218051   
                  2        0.9218051        0.9218051   
                  3        1.3820932        1.3820932   
                  4        2.7653392        2.7653392   
                  5        5.1801941        2.8054757   
                  6        5.1801941        5.1801941   
                  7        5.6093698        5.1801941   
                  8        8.1558663        5.1912743   
      B(N2)       1        0.9219600        0.9219600   
                  2        0.9219600        0.9219600   
                  3        1.3829550        1.3829550   
                  4        2.7678745        2.7678745   
                  5        5.1887039        2.8048037   
                  6        5.1887039        5.1829913   
                  7        5.6126155        5.1829913   
                  8        8.1819000        5.1887039   

                    $\mathrm{dist}(\lambda_j^{\rm half},\sigma^{\rm full}_h)$  \
field formulation j                                                             
b2    B(N1)       1                                           1.43e-14          
                  2                                           2.00e-14          
                  3                                           2.49e-14          
                  4                                           2.22e-14          
                  5                                           3.55e-14          
                  6                                           3.55e-15          
                  7                                           4.53e-14          
                  8                                           8.88e-15          
      B(N2)       1                                           7.15e-14          
                  2                                           4.82e-14          
                  3                                           5.64e-14          
                  4                                           4.26e-14          
                  5                                           2.18e-13          
                  6                                           2.77e-13          
                  7                                           1.38e-13          
                  8                                           2.97e-13          
b3    B(N1)       1                                           8.77e-15          
                  2                                           5.88e-15          
                  3                                           1.22e-14          
                  4                                           1.24e-14          
                  5                                           1.79e-14          
                  6                                           1.79e-14          
                  7                                           7.99e-15          
                  8                                     

## Notes

(a) Parameter re-runs: none. Every code cell is a verbatim copy of `full_hodge_beta_regularity.ipynb` with its stored outputs.

(b) Caveats (reported, not fixed):
- Table 4.8's $\kappa_1(A - \tau M)$ row ($1.395\times10^5$, $1.005\times10^5$) is the MUMPS `COND1` componentwise estimate; the notebook's normwise "Hager $\kappa_1$" column for the same solves is $1.703\times10^8$, $2.920\times10^8$.
- Table 4.9 is the `B(N1)` block of the stored output; the cell also solves and tabulates `B(N2)`, which the thesis omits.
- Table 4.9's rows interleave half and full values by matching (the thesis `--` entries are the values flagged *new in full*); the notebook lists both columns by index $j$.

(c) Dropped cells from the source notebook: 0, 4, 6, 8, 11, 13, 18, 25, 28, 32, 34 (markdown); 1, 3 (LaTeX/matplotlib figure styling and plotting helpers; no figure in this section); 10 (field/divergence panels `fhbr_field_*`, `fhbr_div_b3`, not a thesis item); 15 (conditioning of the Table 4.9 solves, not in the thesis); 16-17 (conditioning under refinement and `fhbr_kappa_b3`, not a thesis item); 19-24 ($h$-refinement of $\beta_3$, $\beta_{11}$, not a thesis item); 26-27 ($p$-refinement, not a thesis item); 29-31 (eigenspace angles, not a thesis item); 33 (`KAPPA_LOG` tally, not a thesis item).
